# EdgeFireUAV — Custom Posture Detector Training (Kaggle)

Trains the 3-class posture model (Lying, Sitting, Standing) using your custom converted dataset.

Before running:
1. Upload your converted `custom_posture_dataset` folder (from `prepare_custom_posture_dataset.py`) as a Kaggle Dataset.
2. Click **Add Input** → add that dataset.
3. Set **Settings → Accelerator → GPU T4 x2**.
4. Run All.

In [ ]:
!pip install -q ultralytics

In [ ]:
import glob, os, yaml

all_yaml = glob.glob('/kaggle/input/**/*.yaml', recursive=True) + glob.glob('/kaggle/input/**/*.yml', recursive=True)
DATA_YAML = all_yaml[0]

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

yaml_dir = os.path.dirname(os.path.abspath(DATA_YAML))
data_cfg['path'] = yaml_dir

fixed_yaml_path = '/kaggle/working/custom_posture_data_fixed.yaml'
with open(fixed_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f)

print('Using dataset yaml:', fixed_yaml_path)
print('Classes:', data_cfg.get('names'))

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=fixed_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs/custom_posture_detector',
    name='train',
    degrees=15.0,
    fliplr=0.5,
)

In [ ]:
metrics = model.val()
print(f'Validation mAP50: {metrics.box.map50:.3f}')
print('Per-class AP:', metrics.box.maps)

In [ ]:
import shutil
best_pt = '/kaggle/working/runs/custom_posture_detector/train/weights/best.pt'
export_model = YOLO(best_pt)
exported_path = export_model.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(str(exported_path), '/kaggle/working/custom_posture_detector.onnx')
print('Exported /kaggle/working/custom_posture_detector.onnx')